In [ ]:
'''
Here we will compare the Statistical Models and Machine Learning Models: 
[SeasonalNaive, MSTL, SeasonalExponentialSmoothing, AutoARIMA and Prophet]
[LGBM, XGB]
Cross Validation Forecasts have been generated, and well be comparing models based on Scaled metrics like:
CV RMSE, CV MAE, and MAPE

'''

In [1]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error
import numpy as np
import pandas as pd 
import polars as pl
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import acf


In [2]:
import plotly.express as px
import plotly.graph_objects as go

In [3]:
dp_Prophet = pl.scan_parquet('cross_validated_data_prophet_only.parquet')

In [4]:
dp_Prophet.schema

C:\Users\tejas\AppData\Local\Temp\ipykernel_2916\3636846125.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  dp_Prophet.schema


Schema([('ds', Datetime(time_unit='ns', time_zone=None)),
        ('trend', Float64),
        ('yhat_lower', Float64),
        ('yhat_upper', Float64),
        ('trend_lower', Float64),
        ('trend_upper', Float64),
        ('additive_terms', Float64),
        ('additive_terms_lower', Float64),
        ('additive_terms_upper', Float64),
        ('yearly', Float64),
        ('yearly_lower', Float64),
        ('yearly_upper', Float64),
        ('multiplicative_terms', Float64),
        ('multiplicative_terms_lower', Float64),
        ('multiplicative_terms_upper', Float64),
        ('yhat', Float64),
        ('unique_id', Int64),
        ('y', Float64),
        ('set', String)])

In [5]:
df_Prophet = dp_Prophet.select(['unique_id','ds','yhat']).collect().to_pandas()
df_Prophet['ds'] = pd.to_datetime(df_Prophet['ds'])
df_Prophet = df_Prophet.rename(columns={'yhat':'Prophet'})

In [6]:
df_Other_Stat_Models = pd.read_parquet('cross_validated_data.parquet')

In [7]:
df_Other_Stat_Models.head(10)

,unique_id,ds,cutoff,y,MSTL,SeasonalNaive,SeasonalES,AutoARIMA
0,1,2017-10-01,2017-09-01,20.8,20.519694,17.9,17.811915,21.801602
1,1,2017-11-01,2017-09-01,19.8,21.092495,18.2,18.388646,21.801602
2,1,2017-12-01,2017-09-01,20.4,20.381647,17.7,17.777967,21.801602
3,1,2018-01-01,2017-09-01,19.6,20.391289,18.7,18.743479,21.801602
4,1,2018-02-01,2017-09-01,18.3,20.590358,17.8,17.809688,21.801602
5,1,2018-03-01,2017-09-01,16.9,20.251725,19.3,19.196359,21.801602
6,1,2018-04-01,2017-09-01,18.7,21.041609,20.3,20.059409,21.801602
7,1,2018-05-01,2017-09-01,19.3,20.715709,19.1,18.907100,21.801602
8,1,2018-06-01,2017-09-01,20.1,21.480160,20.1,19.939382,21.801602
9,1,2018-07-01,2017-09-01,19.0,21.158055,20.8,20.503492,21.801602


In [8]:
df_ML_Models = pd.read_parquet('ml_models_cross_validation.parquet')
df_ML_Models.head()

,unique_id,ds,cutoff,y,LGBMRegressor,XGBRegressor,RandomForestRegressor
0,1,2017-10-01,2017-09-01,20.8,20.429164,20.710810,21.030
1,1,2017-11-01,2017-09-01,19.8,20.175507,20.624928,20.563
2,1,2017-12-01,2017-09-01,20.4,19.990353,19.977713,20.185
3,1,2018-01-01,2017-09-01,19.6,20.048320,19.616003,20.014
4,1,2018-02-01,2017-09-01,18.3,20.099906,20.044794,20.164


In [9]:
df_Full_Set = df_Other_Stat_Models.merge(df_Prophet, on=['unique_id','ds'], how='left')
df_Full_Set = df_Full_Set.merge(df_ML_Models.drop(['y','cutoff'], axis=1), on=['unique_id','ds'], how='left')
df_Full_Set.head()

,unique_id,ds,cutoff,y,MSTL,SeasonalNaive,SeasonalES,AutoARIMA,Prophet,LGBMRegressor,XGBRegressor,RandomForestRegressor
0,1,2017-10-01,2017-09-01,20.8,20.519694,17.9,17.811915,21.801602,19.074046,20.429164,20.710810,21.030
1,1,2017-11-01,2017-09-01,19.8,21.092495,18.2,18.388646,21.801602,18.870374,20.175507,20.624928,20.563
2,1,2017-12-01,2017-09-01,20.4,20.381647,17.7,17.777967,21.801602,18.989494,19.990353,19.977713,20.185
3,1,2018-01-01,2017-09-01,19.6,20.391289,18.7,18.743479,21.801602,19.022341,20.048320,19.616003,20.014
4,1,2018-02-01,2017-09-01,18.3,20.590358,17.8,17.809688,21.801602,18.982241,20.099906,20.044794,20.164


In [14]:
df_Full_Set.columns

Index(['unique_id', 'ds', 'cutoff', 'y', 'MSTL', 'SeasonalNaive', 'SeasonalES',
       'AutoARIMA', 'Prophet', 'LGBMRegressor', 'XGBRegressor',
       'RandomForestRegressor'],
      dtype='object')

In [15]:
Models_Predictions = [
    'MSTL', 'SeasonalNaive', 'SeasonalES',
       'AutoARIMA', 'Prophet', 'LGBMRegressor', 'XGBRegressor', 'RandomForestRegressor'

]

In [16]:
cutoffs = sorted(df_Full_Set['cutoff'].unique().tolist())
cutoffs

[Timestamp('2017-09-01 00:00:00'), Timestamp('2021-09-01 00:00:00')]

In [17]:
cutoff_dicts = {
    cutoffs[0]: 'Validation Set',
    cutoffs[1]: 'Test Set'
}

In [18]:
df_Full_Set['set'] = df_Full_Set['cutoff'].map(cutoff_dicts)

In [19]:
unique_ids = df_Full_Set['unique_id'].unique().tolist()

In [20]:
len(unique_ids)

554

In [21]:
cutoff_dicts

{Timestamp('2017-09-01 00:00:00'): 'Validation Set',
 Timestamp('2021-09-01 00:00:00'): 'Test Set'}

In [22]:
df_Full_Set.head()

,unique_id,ds,cutoff,y,MSTL,SeasonalNaive,SeasonalES,AutoARIMA,Prophet,LGBMRegressor,XGBRegressor,RandomForestRegressor,set
0,1,2017-10-01,2017-09-01,20.8,20.519694,17.9,17.811915,21.801602,19.074046,20.429164,20.710810,21.030,Validation Set
1,1,2017-11-01,2017-09-01,19.8,21.092495,18.2,18.388646,21.801602,18.870374,20.175507,20.624928,20.563,Validation Set
2,1,2017-12-01,2017-09-01,20.4,20.381647,17.7,17.777967,21.801602,18.989494,19.990353,19.977713,20.185,Validation Set
3,1,2018-01-01,2017-09-01,19.6,20.391289,18.7,18.743479,21.801602,19.022341,20.048320,19.616003,20.014,Validation Set
4,1,2018-02-01,2017-09-01,18.3,20.590358,17.8,17.809688,21.801602,18.982241,20.099906,20.044794,20.164,Validation Set


In [23]:
def smape(y_true, y_pred):
    return 100 * np.mean(np.abs(y_pred - y_true) / ((np.abs(y_true) + np.abs(y_pred)) / 2))

In [24]:
Metrics = []
for uid in unique_ids:
    Sample = df_Full_Set[df_Full_Set['unique_id']==uid]

    for c, cuttset in cutoff_dicts.items():

        SampleReduced = Sample[Sample['cutoff']==c]

        for model in Models_Predictions:
            SampleReduced = SampleReduced.dropna(subset=['y',model])
            avgY = SampleReduced['y'].mean()
            rmse = root_mean_squared_error(SampleReduced['y'], SampleReduced[model])
            mae = mean_absolute_error(SampleReduced['y'], SampleReduced[model])
            sMAPE = smape(SampleReduced['y'], SampleReduced[model])
            Metrics.append({
            'unique_id':uid,
            'Model': model,
            'CV RMSE': 100 * (rmse/avgY),
            'CV MAE': 100 * (mae/avgY),
            'sMAPE': sMAPE,
            'Set': cuttset
            
        })

df_Metrics = pd.DataFrame(Metrics)
df_Metrics



,unique_id,Model,CV RMSE,CV MAE,sMAPE,Set
0,1,MSTL,8.762253,7.093292,7.134329,Validation Set
1,1,SeasonalNaive,10.822396,8.940159,9.008189,Validation Set
2,1,SeasonalES,10.559641,8.778808,8.867229,Validation Set
3,1,AutoARIMA,13.571335,11.304901,11.079248,Validation Set
4,1,Prophet,7.701140,6.069473,6.128586,Validation Set
...,...,...,...,...,...,...
8859,554,AutoARIMA,23.152937,17.746790,17.525868,Test Set
8860,554,Prophet,21.694592,16.664746,16.395793,Test Set
8861,554,LGBMRegressor,22.476786,18.450284,18.140773,Test Set
8862,554,XGBRegressor,20.231738,16.599549,16.254948,Test Set


In [ ]:
df_Metrics.head(16).to_csv('tables/Metrics_Set.csv',index=False)

In [16]:
#Plotting Error Distributions for Metrics

In [28]:
color_discrete_map = {
    'SeasonalNaive': '#474340',
    'MSTL': "#B87C39",
    'SeasonalES': "#321975",
    'AutoARIMA': "#5DBAC8",
    'Prophet': "#103C6B",
    'LGBMRegressor': "#1D6E4E",
    'XGBRegressor':"#6E1D6E",
    'RandomForestRegressor': "#751D12"
}

In [29]:
Box = px.box(
    title='CV RMSE Distribution, Color Coded by Model, Split by Sets - Outliers Removed',
    data_frame=df_Metrics[df_Metrics['CV RMSE'] <= 100],
    x='Model',
    y='CV RMSE',
    width=1500,
    height=900,
    color='Model',
    color_discrete_map=color_discrete_map,
    template='ggplot2',
    facet_col='Set',
    points=False,
)

Box.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=12),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#CDD0D4", 
)

Box.show()

In [ ]:
Box.write_image('figures/Baseline_Comparison_Without_outliers.svg')

In [72]:
Box = px.box(
    title='sMAPE Distribution, Color Coded by Model, Split by Sets - Outliers Removed',
    data_frame=df_Metrics[df_Metrics['sMAPE'] <= 100],
    x='Model',
    y='sMAPE',
    width=1500,
    height=900,
    color='Model',
    color_discrete_map=color_discrete_map,
    template='ggplot2',
    facet_col='Set',
    points=False,
)

Box.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=12),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#CDD0D4", 
)

Box.show()

In [ ]:
Box.write_image('figures/Baseline_Comparison_Without_outliers_smape.svg')

In [87]:
Box = px.box(
    title='CV RMSE Distribution, Color Coded by Model, Split by Sets, Outliers Included',
    data_frame=df_Metrics[(df_Metrics['Model'].isin(['SeasonalNaive','Prophet','XGBRegressor','RandomForestRegressor','AutoARIMA']))],
    y='CV RMSE',
    x='Model',
    width=1500,
    height=900,
    color='Model',
    color_discrete_map=color_discrete_map,
    template='ggplot2',
    facet_col='Set'
)

Box.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=12),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#CDD0D4", 
)

Box.show()

In [ ]:
Box.write_image('figures/Interesting_models_cv_rmse.svg')

In [31]:
df_CV_RMSE = df_Metrics.groupby('Model')['CV RMSE'].describe().round(3).reset_index()
df_SMAPE = df_Metrics.groupby('Model')['sMAPE'].describe().round(3).reset_index()

In [32]:
df_CV_RMSE

,Model,count,mean,std,min,25%,50%,75%,max
0,AutoARIMA,1108.0,18.744,10.374,2.343,11.266,16.609,23.655,88.748
1,LGBMRegressor,1108.0,22.606,23.241,2.801,11.734,17.532,26.034,379.027
2,MSTL,1108.0,20.828,12.994,2.364,12.004,17.826,25.927,139.051
3,Prophet,1108.0,18.804,10.203,2.253,11.344,16.722,23.967,76.353
4,RandomForestRegressor,1108.0,19.815,11.369,2.368,11.793,17.235,25.328,83.127
5,SeasonalES,1108.0,21.487,11.461,2.311,13.066,19.063,27.094,90.734
6,SeasonalNaive,1108.0,22.161,12.020,2.318,13.358,19.648,27.860,97.935
7,XGBRegressor,1108.0,20.583,12.917,3.091,12.024,17.876,25.846,150.377


In [33]:
df_SMAPE

,Model,count,mean,std,min,25%,50%,75%,max
0,AutoARIMA,1108.0,15.636,8.903,1.366,9.201,13.823,19.637,70.144
1,LGBMRegressor,1108.0,17.643,12.860,2.181,9.551,14.464,21.565,126.020
2,MSTL,1108.0,17.806,11.729,1.558,9.962,14.871,22.066,81.433
3,Prophet,1108.0,15.887,9.204,1.565,9.125,13.882,20.205,82.513
4,RandomForestRegressor,1108.0,16.406,9.381,1.678,9.519,14.154,21.001,60.511
5,SeasonalES,1108.0,18.292,10.014,1.591,10.899,16.015,22.921,62.443
6,SeasonalNaive,1108.0,18.908,10.524,1.543,11.136,16.430,23.823,65.579
7,XGBRegressor,1108.0,16.952,9.971,2.412,9.808,14.604,21.838,81.958


In [ ]:
df_CV_RMSE.to_csv('tables/CV_RMSE_Baseline.csv',index=False)
df_SMAPE.to_csv('tables/SMAPE_Baseline.csv', index=False)

In [ ]:
#Prophet seema to perform better than SeasonalNaive

In [34]:
ts_sample = df_Full_Set[df_Full_Set['unique_id']==1]
ts_sample

,unique_id,ds,cutoff,y,MSTL,SeasonalNaive,SeasonalES,AutoARIMA,Prophet,LGBMRegressor,XGBRegressor,RandomForestRegressor,set
0,1,2017-10-01,2017-09-01,20.8,20.519694,17.9,17.811915,21.801602,19.074046,20.429164,20.710810,21.030,Validation Set
1,1,2017-11-01,2017-09-01,19.8,21.092495,18.2,18.388646,21.801602,18.870374,20.175507,20.624928,20.563,Validation Set
2,1,2017-12-01,2017-09-01,20.4,20.381647,17.7,17.777967,21.801602,18.989494,19.990353,19.977713,20.185,Validation Set
3,1,2018-01-01,2017-09-01,19.6,20.391289,18.7,18.743479,21.801602,19.022341,20.048320,19.616003,20.014,Validation Set
4,1,2018-02-01,2017-09-01,18.3,20.590358,17.8,17.809688,21.801602,18.982241,20.099906,20.044794,20.164,Validation Set
...,...,...,...,...,...,...,...,...,...,...,...,...,...
140,1,2025-07-01,2021-09-01,25.7,22.736496,21.7,21.737950,22.620160,22.479831,23.747378,26.142811,19.521,Test Set
141,1,2025-08-01,2021-09-01,24.0,22.456065,21.7,21.571176,22.620160,22.491016,23.731358,26.044661,19.462,Test Set
142,1,2025-08-01,2021-09-01,24.0,22.456065,21.7,21.571176,22.620160,22.354337,23.731358,26.044661,19.462,Test Set
143,1,2025-09-01,2021-09-01,24.3,22.646962,22.9,22.714445,22.620160,22.476077,23.208512,25.270260,19.702,Test Set


In [29]:
ts_sample['SN_Residuals'] = ts_sample['y'] - ts_sample['SeasonalNaive']
ts_sample['Prophet_Residuals'] = ts_sample['y'] - ts_sample['Prophet']

C:\Users\tejas\AppData\Local\Temp\ipykernel_46360\3411981112.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ts_sample['SN_Residuals'] = ts_sample['y'] - ts_sample['SeasonalNaive']
C:\Users\tejas\AppData\Local\Temp\ipykernel_46360\3411981112.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ts_sample['Prophet_Residuals'] = ts_sample['y'] - ts_sample['Prophet']


In [ ]:
# Compute ACF
nlags = 24  # 2 years worth of lags
residuals = ts_sample['SN_Residuals']
acf_vals = acf(residuals, nlags=nlags)

# Create subplot layout: 2 rows, 2 columns for row 2
fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=False,
    column_widths=[0.5, 0.5],
    row_heights=[0.6, 0.4],  # first row taller
    specs=[[{"colspan": 2}, None], [{"type": "xy"}, {"type": "xy"}]],
    subplot_titles=("Residuals over Time", "Residuals Histogram", "Residuals ACF"),
    vertical_spacing=0.10
)

fig.add_trace(
    go.Scatter(
        x=ts_sample['ds'],
        y=residuals,
        mode='lines',
        line=dict(color=color_discrete_map['SeasonalNaive'], width=2.5),
        name='Residuals'
    ),
    row=1, col=1
)

# Horizontal line at 0
fig.add_hline(y=0, line_dash="dash", line_color="black", row=1, col=1)

fig.add_trace(
    go.Histogram(
        x=residuals,
        nbinsx=30,
        name='Residuals',
        marker_color=color_discrete_map['SeasonalNaive'],
        marker_line_color='black',   # border color
        marker_line_width=1          # border width
    ),
    row=2, col=1
)

fig.add_trace(
    go.Bar(
        x=np.arange(len(acf_vals)),
        y=acf_vals,
        name='ACF',
        marker_color=color_discrete_map['SeasonalNaive'],
    ),
    row=2, col=2
)

# Add confidence intervals for ACF
conf_int = 1.96/np.sqrt(len(residuals))
fig.add_hline(y=conf_int, line_dash="dot", line_color="red", row=2, col=2)
fig.add_hline(y=-conf_int, line_dash="dot", line_color="red", row=2, col=2)

# Layout adjustments
fig.update_layout(
    height=900,
    width=1000,
    title_text="Seasonal Naive Residual Analysis for Unique Series 1",
    showlegend=False,
)

fig.show()

In [ ]:
# Compute ACF
nlags = 24  # 2 years worth of lags
residuals = ts_sample['Prophet_Residuals']
acf_vals = acf(residuals, nlags=nlags)

# Create subplot layout: 2 rows, 2 columns for row 2
fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=False,
    column_widths=[0.5, 0.5],
    row_heights=[0.6, 0.4],  # first row taller
    specs=[[{"colspan": 2}, None], [{"type": "xy"}, {"type": "xy"}]],
    subplot_titles=("Residuals over Time", "Residuals Histogram", "Residuals ACF"),
    vertical_spacing=0.10
)

fig.add_trace(
    go.Scatter(
        x=ts_sample['ds'],
        y=residuals,
        mode='lines',
        line=dict(color=color_discrete_map['Prophet'], width=2.5),
        name='Residuals'
    ),
    row=1, col=1
)

# Horizontal line at 0
fig.add_hline(y=0, line_dash="dash", line_color="black", row=1, col=1)

fig.add_trace(
    go.Histogram(
        x=residuals,
        nbinsx=30,
        name='Residuals',
        marker_color=color_discrete_map['Prophet'],
        marker_line_color='black',   # border color
        marker_line_width=1          # border width
    ),
    row=2, col=1
)

fig.add_trace(
    go.Bar(
        x=np.arange(len(acf_vals)),
        y=acf_vals,
        name='ACF',
        marker_color=color_discrete_map['Prophet'],
    ),
    row=2, col=2
)

# Add confidence intervals for ACF
conf_int = 1.96/np.sqrt(len(residuals))
fig.add_hline(y=conf_int, line_dash="dot", line_color="red", row=2, col=2)
fig.add_hline(y=-conf_int, line_dash="dot", line_color="red", row=2, col=2)

# Layout adjustments
fig.update_layout(
    height=900,
    width=1000,
    title_text="Prophet Residual Analysis for Unique Series 1",
    showlegend=False,
)

fig.show()

Checking if Prophet is actuall Significantly better than Seasonal Naives

In [42]:
from scipy.stats import ttest_rel
from scipy.stats import wilcoxon

In [36]:
df_Metrics

,unique_id,Model,CV RMSE,CV MAE,sMAPE,Set
0,1,MSTL,8.762253,7.093292,7.134329,Validation Set
1,1,SeasonalNaive,10.822396,8.940159,9.008189,Validation Set
2,1,SeasonalES,10.559641,8.778808,8.867229,Validation Set
3,1,AutoARIMA,13.571335,11.304901,11.079248,Validation Set
4,1,Prophet,7.701140,6.069473,6.128586,Validation Set
...,...,...,...,...,...,...
8859,554,AutoARIMA,23.152937,17.746790,17.525868,Test Set
8860,554,Prophet,21.694592,16.664746,16.395793,Test Set
8861,554,LGBMRegressor,22.476786,18.450284,18.140773,Test Set
8862,554,XGBRegressor,20.231738,16.599549,16.254948,Test Set


In [37]:
df_Metrics_Reduced = df_Metrics[df_Metrics['Model'].isin(['Prophet','XGBRegressor','RandomForestRegressor','SeasonalNaive'])][['unique_id','Set','Model','CV RMSE']]
df_Metrics_Reduced

,unique_id,Set,Model,CV RMSE
1,1,Validation Set,SeasonalNaive,10.822396
4,1,Validation Set,Prophet,7.701140
6,1,Validation Set,XGBRegressor,9.686958
7,1,Validation Set,RandomForestRegressor,9.914924
9,1,Test Set,SeasonalNaive,10.363721
...,...,...,...,...
8855,554,Validation Set,RandomForestRegressor,24.952817
8857,554,Test Set,SeasonalNaive,24.231834
8860,554,Test Set,Prophet,21.694592
8862,554,Test Set,XGBRegressor,20.231738


In [38]:
print(df_Metrics_Reduced)

      unique_id             Set                  Model    CV RMSE
1             1  Validation Set          SeasonalNaive  10.822396
4             1  Validation Set                Prophet   7.701140
6             1  Validation Set           XGBRegressor   9.686958
7             1  Validation Set  RandomForestRegressor   9.914924
9             1        Test Set          SeasonalNaive  10.363721
...         ...             ...                    ...        ...
8855        554  Validation Set  RandomForestRegressor  24.952817
8857        554        Test Set          SeasonalNaive  24.231834
8860        554        Test Set                Prophet  21.694592
8862        554        Test Set           XGBRegressor  20.231738
8863        554        Test Set  RandomForestRegressor  20.931443

[4432 rows x 4 columns]


In [39]:
df_pivot = df_Metrics_Reduced.pivot_table(
    index=['unique_id','Set'],
    columns='Model',
    values='CV RMSE',
).reset_index()

df_pivot

Model,unique_id,Set,Prophet,RandomForestRegressor,SeasonalNaive,XGBRegressor
0,1,Test Set,8.194818,12.243594,10.363721,13.302504
1,1,Validation Set,7.701140,9.914924,10.822396,9.686958
2,2,Test Set,28.700309,39.214398,38.388846,24.452057
3,2,Validation Set,22.624441,31.754478,37.871668,40.164491
4,3,Test Set,9.823198,8.820790,10.230429,7.245939
...,...,...,...,...,...,...
1103,552,Validation Set,10.565091,15.092675,19.362929,19.860811
1104,553,Test Set,19.648408,10.250570,11.580427,10.676145
1105,553,Validation Set,11.310701,12.400921,13.505827,11.990959
1106,554,Test Set,21.694592,20.931443,24.231834,20.231738


In [ ]:
df_pivot.head(10).to_csv('tables/Paired_testing.csv',index=False)

In [93]:
df_pivot['Prophet vs. SeasonalNaive'] = df_pivot['Prophet'] - df_pivot['SeasonalNaive']
df_pivot['RF vs. SeasonalNaive'] = df_pivot['RandomForestRegressor'] - df_pivot['SeasonalNaive']
df_pivot['XGB vs. SeasonalNaive'] = df_pivot['XGBRegressor'] - df_pivot['SeasonalNaive']
df_pivot

Model,unique_id,Set,Prophet,RandomForestRegressor,SeasonalNaive,XGBRegressor,Prophet vs. SeasonalNaive,RF vs. SeasonalNaive,XGB vs. SeasonalNaive
0,1,Test Set,8.194818,12.243594,10.363721,13.302504,-2.168903,1.879872,2.938783
1,1,Validation Set,7.701140,9.914924,10.822396,9.686958,-3.121255,-0.907472,-1.135438
2,2,Test Set,28.700309,39.214398,38.388846,24.452057,-9.688537,0.825552,-13.936788
3,2,Validation Set,22.624441,31.754478,37.871668,40.164491,-15.247227,-6.117190,2.292823
4,3,Test Set,9.823198,8.820790,10.230429,7.245939,-0.407231,-1.409639,-2.984490
...,...,...,...,...,...,...,...,...,...
1103,552,Validation Set,10.565091,15.092675,19.362929,19.860811,-8.797838,-4.270254,0.497882
1104,553,Test Set,19.648408,10.250570,11.580427,10.676145,8.067981,-1.329857,-0.904282
1105,553,Validation Set,11.310701,12.400921,13.505827,11.990959,-2.195127,-1.104907,-1.514869
1106,554,Test Set,21.694592,20.931443,24.231834,20.231738,-2.537242,-3.300391,-4.000096


In [112]:
Difference_Distribution = px.histogram(
    title='Prophet (CV RMSE) - SeasonalNaive (CV RMSE)',
    data_frame=df_pivot,
    x='Prophet vs. SeasonalNaive',
    nbins=50,
    width=1000,
    height=900,
    facet_col='Set',
    marginal='box',
    histnorm='percent',
    
)

# Change bar color and add borders
Difference_Distribution.update_traces(
    marker_color=color_discrete_map['Prophet'],     # histogram color
    marker_line_color='black',  # border color
    marker_line_width=1         # border width
)

# Add vertical line at x = 0
Difference_Distribution.add_vline(
    x=0,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=15),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
            yaxis=dict(
                title='Proportion of Series',
                title_font=dict(size=18),  # Set y-axis title font size
                tickfont=dict(size=12)  # Set y-axis tick font size
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#B4C2D4", 
)

Difference_Distribution.show()

In [ ]:
Difference_Distribution.write_image('figures/Prophet_Vs_SeasonalNaive.svg')

In [133]:
Difference_Distribution_Cumulative = px.histogram(
    title='Prophet (CV RMSE) - SeasonalNaive (CV RMSE) - CDF',
    data_frame=df_pivot,
    x='Prophet vs. SeasonalNaive',
    nbins=50,
    width=1500,
    height=900,
    facet_col='Set',
    histnorm='percent',
    cumulative=True,
    
)

# Change bar color and add borders
Difference_Distribution_Cumulative.update_traces(
    marker_color=color_discrete_map['Prophet'],     # histogram color
    marker_line_color='black',  # border color
    marker_line_width=1         # border width
)

# Add vertical line at x = 0
Difference_Distribution_Cumulative.add_vline(
    x=0,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution_Cumulative.add_hline(
    y=80,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution_Cumulative.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=15),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
            yaxis=dict(
                title='Cumulative Proportion',
                title_font=dict(size=18),  # Set y-axis title font size
                tickfont=dict(size=12)  # Set y-axis tick font size
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#B4C2D4", 
)

Difference_Distribution_Cumulative.show()

In [ ]:
Difference_Distribution_Cumulative.write_image('figures/Prophet_vs_SeasonalNaive_CDF.svg')

In [147]:
Difference_Distribution = px.histogram(
    title='Random Forest (CV RMSE) - SeasonalNaive (CV RMSE)',
    data_frame=df_pivot,
    x='RF vs. SeasonalNaive',
    nbins=30,
    width=1000,
    height=900,
    facet_col='Set',
     marginal='box',
    histnorm='percent',
)

# Change bar color and add borders
Difference_Distribution.update_traces(
    marker_color=color_discrete_map['RandomForestRegressor'],     # histogram color
    marker_line_color='black',  # border color
    marker_line_width=1         # border width
)

# Add vertical line at x = 0
Difference_Distribution.add_vline(
    x=0,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=15),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
            yaxis=dict(
                title='Proportion of Series',
                title_font=dict(size=18),  # Set y-axis title font size
                tickfont=dict(size=12)  # Set y-axis tick font size
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#B4C2D4", 
)

Difference_Distribution.show()

In [148]:
Difference_Distribution_Cumulative = px.histogram(
    title='Randon Forest (CV RMSE) - SeasonalNaive (CV RMSE) - CDF',
    data_frame=df_pivot,
    x='RF vs. SeasonalNaive',
    nbins=50,
    width=1500,
    height=900,
    facet_col='Set',
    histnorm='percent',
    cumulative=True,
    
)

# Change bar color and add borders
Difference_Distribution_Cumulative.update_traces(
    marker_color=color_discrete_map['RandomForestRegressor'],     # histogram color
    marker_line_color='black',  # border color
    marker_line_width=1         # border width
)

# Add vertical line at x = 0
Difference_Distribution_Cumulative.add_vline(
    x=0,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution_Cumulative.add_hline(
    y=80,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution_Cumulative.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=15),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
            yaxis=dict(
                title='Cumulative Proportion',
                title_font=dict(size=18),  # Set y-axis title font size
                tickfont=dict(size=12)  # Set y-axis tick font size
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#B4C2D4", 
)

Difference_Distribution_Cumulative.show()

In [ ]:
Difference_Distribution.write_image('figures/RandomForest_vs_SeasonalNaive.svg')

In [ ]:
Difference_Distribution_Cumulative.write_image('figures/RandomForest_vs_SeasonalNaive_CDF.svg')

In [151]:
Difference_Distribution = px.histogram(
    title='XGB (CV RMSE) - SeasonalNaive (CV RMSE)',
    data_frame=df_pivot,
    x='XGB vs. SeasonalNaive',
    nbins=50,
    width=1000,
    height=900,
    facet_col='Set',
     marginal='box',
    histnorm='percent',
)

# Change bar color and add borders
Difference_Distribution.update_traces(
    marker_color=color_discrete_map['XGBRegressor'],     # histogram color
    marker_line_color='black',  # border color
    marker_line_width=1         # border width
)

# Add vertical line at x = 0
Difference_Distribution.add_vline(
    x=0,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=15),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
            yaxis=dict(
                title='Proportion of Series',
                title_font=dict(size=18),  # Set y-axis title font size
                tickfont=dict(size=12)  # Set y-axis tick font size
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#B4C2D4", 
)

Difference_Distribution.show()

In [152]:
Difference_Distribution_Cumulative = px.histogram(
    title='XGB (CV RMSE) - SeasonalNaive (CV RMSE) - CDF',
    data_frame=df_pivot,
    x='XGB vs. SeasonalNaive',
    nbins=50,
    width=1500,
    height=900,
    facet_col='Set',
    histnorm='percent',
    cumulative=True,
    
)

# Change bar color and add borders
Difference_Distribution_Cumulative.update_traces(
    marker_color=color_discrete_map['XGBRegressor'],     # histogram color
    marker_line_color='black',  # border color
    marker_line_width=1         # border width
)

# Add vertical line at x = 0
Difference_Distribution_Cumulative.add_vline(
    x=0,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution_Cumulative.add_hline(
    y=80,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution_Cumulative.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=15),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
            yaxis=dict(
                title='Cumulative Proportion',
                title_font=dict(size=18),  # Set y-axis title font size
                tickfont=dict(size=12)  # Set y-axis tick font size
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#B4C2D4", 
)

Difference_Distribution_Cumulative.show()

In [ ]:
Difference_Distribution.write_image('figures/XGB_vs_SeasonalNaive.svg')

In [ ]:
Difference_Distribution_Cumulative.write_image('figures/XGB_vs_SeasonalNaive_CDF.svg')

In [155]:
SetsTesting = df_pivot['Set'].unique()

In [156]:
SetsTesting

array(['Test Set', 'Validation Set'], dtype=object)

In [157]:
df_pivot

Model,unique_id,Set,Prophet,RandomForestRegressor,SeasonalNaive,XGBRegressor,Prophet vs. SeasonalNaive,RF vs. SeasonalNaive,XGB vs. SeasonalNaive
0,1,Test Set,8.194818,12.243594,10.363721,13.302504,-2.168903,1.879872,2.938783
1,1,Validation Set,7.701140,9.914924,10.822396,9.686958,-3.121255,-0.907472,-1.135438
2,2,Test Set,28.700309,39.214398,38.388846,24.452057,-9.688537,0.825552,-13.936788
3,2,Validation Set,22.624441,31.754478,37.871668,40.164491,-15.247227,-6.117190,2.292823
4,3,Test Set,9.823198,8.820790,10.230429,7.245939,-0.407231,-1.409639,-2.984490
...,...,...,...,...,...,...,...,...,...
1103,552,Validation Set,10.565091,15.092675,19.362929,19.860811,-8.797838,-4.270254,0.497882
1104,553,Test Set,19.648408,10.250570,11.580427,10.676145,8.067981,-1.329857,-0.904282
1105,553,Validation Set,11.310701,12.400921,13.505827,11.990959,-2.195127,-1.104907,-1.514869
1106,554,Test Set,21.694592,20.931443,24.231834,20.231738,-2.537242,-3.300391,-4.000096


In [158]:
ModelsLeft = ['Prophet','RandomForestRegressor','XGBRegressor']

In [159]:
test = []
for set in SetsTesting:
    for model in ModelsLeft:
        df_pivot_reduced = df_pivot[df_pivot['Set']==set]

        t_stat, p_val = ttest_rel(df_pivot_reduced[model], df_pivot_reduced['SeasonalNaive'], alternative='less')
    
        wilcoxon_result = wilcoxon(
        df_pivot_reduced[model],
        df_pivot_reduced['SeasonalNaive'],
        alternative='less'
        )
    
        test.append({
            'Model':model,
            'Set': set,
            't-statistic':t_stat,
            't-test p': p_val,
            'Wilcoxon p': wilcoxon_result.pvalue
        })

test

[{'Model': 'Prophet',
  'Set': 'Test Set',
  't-statistic': -13.427320748531677,
  't-test p': 4.424876848039668e-36,
  'Wilcoxon p': 3.774747567907076e-40},
 {'Model': 'RandomForestRegressor',
  'Set': 'Test Set',
  't-statistic': -12.847767611638597,
  't-test p': 1.5143262006212238e-33,
  'Wilcoxon p': 1.5198716096585548e-42},
 {'Model': 'XGBRegressor',
  'Set': 'Test Set',
  't-statistic': -6.7973504023649,
  't-test p': 1.3835294638972156e-11,
  'Wilcoxon p': 3.316674520418026e-28},
 {'Model': 'Prophet',
  'Set': 'Validation Set',
  't-statistic': -12.94516679719711,
  't-test p': 5.732589357223096e-34,
  'Wilcoxon p': 3.9924982140347416e-37},
 {'Model': 'RandomForestRegressor',
  'Set': 'Validation Set',
  't-statistic': -8.015345122030745,
  't-test p': 3.28091568494058e-15,
  'Wilcoxon p': 1.5489354852387116e-19},
 {'Model': 'XGBRegressor',
  'Set': 'Validation Set',
  't-statistic': -2.405939117604835,
  't-test p': 0.008229056724665878,
  'Wilcoxon p': 6.771710471144187e-11}]

In [160]:
df_Hypothesis_Tests = pd.DataFrame(test)
df_Hypothesis_Tests

,Model,Set,t-statistic,t-test p,Wilcoxon p
0,Prophet,Test Set,-13.427321,4.424877e-36,3.774748e-40
1,RandomForestRegressor,Test Set,-12.847768,1.514326e-33,1.519872e-42
2,XGBRegressor,Test Set,-6.797350,1.383529e-11,3.316675e-28
3,Prophet,Validation Set,-12.945167,5.732589e-34,3.992498e-37
4,RandomForestRegressor,Validation Set,-8.015345,3.280916e-15,1.548935e-19
5,XGBRegressor,Validation Set,-2.405939,8.229057e-03,6.771710e-11


In [162]:
print(df_Hypothesis_Tests)

                   Model             Set  t-statistic      t-test p  \
0                Prophet        Test Set   -13.427321  4.424877e-36   
1  RandomForestRegressor        Test Set   -12.847768  1.514326e-33   
2           XGBRegressor        Test Set    -6.797350  1.383529e-11   
3                Prophet  Validation Set   -12.945167  5.732589e-34   
4  RandomForestRegressor  Validation Set    -8.015345  3.280916e-15   
5           XGBRegressor  Validation Set    -2.405939  8.229057e-03   

     Wilcoxon p  
0  3.774748e-40  
1  1.519872e-42  
2  3.316675e-28  
3  3.992498e-37  
4  1.548935e-19  
5  6.771710e-11  


In [ ]:
df_Hypothesis_Tests.to_csv('figures/Models_vs_SeasonalNaive_Hypothesis_Testing.csv',index=False)